In [ ]:
# ==========================================================
# BLOQUE DE PORTABILIDAD: Colab + Drive + Local
# ==========================================================
import os
import sys

# 1. Detectar si estamos en Google Colab
EN_COLAB = 'google.colab' in sys.modules

# 2. CONFIGURACIÓN IMPORTANTE: Nombra tu carpeta en Drive así:
CARPETA_DRIVE = 'Proyecto_IA'

if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Entramos a la subcarpeta notebooks para que las rutas ../data y ../outputs funcionen igual que en tu PC
    ruta_notebooks = f'/content/drive/MyDrive/{CARPETA_DRIVE}/notebooks'
    os.chdir(ruta_notebooks)
    print(f"✅ Colab conectado. Directorio actual: {os.getcwd()} (Estás en la carpeta notebooks)")
else:
    print(f"✅ Ejecutando en Local. Asegúrate de estar en la carpeta notebooks del proyecto.")
    print(f"Directorio actual: {os.getcwd()}")
# FORZAR RUTA (Solo para Colab)
if EN_COLAB:
    import os
    os.chdir('/content/drive/MyDrive/Proyecto_IA/notebooks')
    print(f"✅ Ruta forzada. Ahora estás en: {os.getcwd()}")
# -*- coding: utf-8 -*-
"""
 MODELOS DE MACHINE LEARNING

FASE 2: ENTRENAMIENTO Y EVALUACIÓN CON SCIKIT-LEARN Y PYTORCH
"""

# 1. IMPORTAR LIBRERÍAS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from scipy.sparse import hstack, csr_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("✅ Librerías cargadas")

# 2. CARGAR DATOS
df = pd.read_csv('../data/dataset_procesado.csv', sep=';', encoding='latin-1')
print(f"✅ Dataset: {df.shape[0]} filas")

# 3. PREPARAR CARACTERÍSTICAS
features_num = ['daily_return', 'intraday_volatility', 'volume_scaled', 'price_range']
X_num = df[features_num].values
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)

tfidf = TfidfVectorizer(max_features=500, stop_words='spanish')
X_text = tfidf.fit_transform(df['text_clean'].values)

X_combined = hstack([csr_matrix(X_num_scaled), X_text])
y = df['Label'].values

print(f"✅ Características combinadas: {X_combined.shape}")

# 4. DIVIDIR DATOS
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)
print(f"✅ Entrenamiento: {len(y_train)} | Prueba: {len(y_test)}")

# 5. REGRESIÓN LOGÍSTICA
print("\n📊 REGRESIÓN LOGÍSTICA")
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)
y_pred_log = log_reg.predict(X_test)
print(f"Precisión: {accuracy_score(y_test, y_pred_log):.4f}")
print(classification_report(y_test, y_pred_log, target_names=['Baja', 'Subida']))

# 6. RANDOM FOREST
print("\n🌲 RANDOM FOREST")
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print(f"Precisión: {accuracy_score(y_test, y_pred_rf):.4f}")
print(classification_report(y_test, y_pred_rf, target_names=['Baja', 'Subida']))

# 7. PYTORCH MLP (solo numéricos)
print("\n PYTORCH MLP")
X_num_train, X_num_test, _, _ = train_test_split(
    X_num_scaled, y, test_size=0.2, random_state=42, stratify=y
)

X_train_t = torch.tensor(X_num_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_num_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

class MLPClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

model_pt = MLPClassifier(input_dim=X_num_train.shape[1])
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_pt.parameters(), lr=0.001)
dataset = TensorDataset(X_train_t, y_train_t)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

print(" Entrenando...")
for epoch in range(50):
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        outputs = model_pt(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

with torch.no_grad():
    outputs_test = model_pt(X_test_t)
    _, predicted = torch.max(outputs_test, 1)
    acc_pt = (predicted == y_test_t).float().mean().item()
print(f"Precisión: {acc_pt:.4f}")

# 8. MATRICES DE CONFUSIÓN
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, (name, y_pred) in enumerate([
    ('Regresión Logística', y_pred_log),
    ('Random Forest', y_pred_rf),
    ('PyTorch MLP', predicted.numpy())
]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
    axes[idx].set_title(name)
plt.tight_layout()
plt.savefig('../outputs/graficos/matrices_confusion.png', dpi=300)
plt.show()
print("✅ Gráfico guardado")

print("\n✅ MODELOS COMPLETADOS")